# Lab 3: Contextual Bandit-Based News Article Recommendation

**`Course`:** Reinforcement Learning Fundamentals  
**`Student Name`:** Akshat  
**`Roll Number`:** U20230093  
**`GitHub Branch`:** akshat_U20230093  

# Imports and Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

from rlcmab_sampler import sampler

# Load Datasets

In [2]:
# Load datasets
news_df = pd.read_csv("data/news_articles.csv")
train_users = pd.read_csv("data/train_users.csv")
test_users = pd.read_csv("data/test_users.csv")

print("=== News Articles ===")
print(f"Shape: {news_df.shape}")
print(f"Columns: {news_df.columns.tolist()}")
print(news_df.head())

print("\n=== Train Users ===")
print(f"Shape: {train_users.shape}")
print(f"Columns: {train_users.columns.tolist()}")
print(train_users.head())

print("\n=== Test Users ===")
print(f"Shape: {test_users.shape}")
print(test_users.head())

=== News Articles ===
Shape: (209527, 6)
Columns: ['link', 'headline', 'category', 'short_description', 'authors', 'date']
                                                link  \
0  https://www.huffpost.com/entry/covid-boosters-...   
1  https://www.huffpost.com/entry/american-airlin...   
2  https://www.huffpost.com/entry/funniest-tweets...   
3  https://www.huffpost.com/entry/funniest-parent...   
4  https://www.huffpost.com/entry/amy-cooper-lose...   

                                            headline   category  \
0  Over 4 Million Americans Roll Up Sleeves For O...  U.S. NEWS   
1  American Airlines Flyer Charged, Banned For Li...  U.S. NEWS   
2  23 Of The Funniest Tweets About Cats And Dogs ...     COMEDY   
3  The Funniest Tweets From Parents This Week (Se...  PARENTING   
4  Woman Who Called Cops On Black Bird-Watcher Lo...  U.S. NEWS   

                                   short_description               authors  \
0  Health experts said it is too early to predict...  Carla

## Data Preprocessing

In this section:
- Handle missing values
- Encode categorical features
- Prepare data for user classification

In [3]:
# --- Data Preprocessing ---

# 1. Check for missing values in user datasets
print("Missing values in train_users:")
print(train_users.isnull().sum())
print("\nMissing values in test_users:")
print(test_users.isnull().sum())

# 2. Define features and target
feature_cols = ['age', 'income', 'clicks', 'purchase_amount']

X_train = train_users[feature_cols].copy()
y_train = train_users['label'].copy()

X_test = test_users[feature_cols].copy()
y_test = test_users['label'].copy()

# 3. Encode the target labels (user1, user2, user3) -> (0, 1, 2)
le = LabelEncoder()
le.fit(y_train)
y_train_enc = le.transform(y_train)
y_test_enc = le.transform(y_test)

print(f"\nLabel mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# 4. Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nTraining set: {X_train_scaled.shape[0]} samples")
print(f"Test set:     {X_test_scaled.shape[0]} samples")
print(f"Features:     {feature_cols}")

Missing values in train_users:
user_id            0
age                0
income             0
clicks             0
purchase_amount    0
label              0
dtype: int64

Missing values in test_users:
user_id            0
age                0
income             0
clicks             0
purchase_amount    0
label              0
dtype: int64

Label mapping: {'user1': np.int64(0), 'user2': np.int64(1), 'user3': np.int64(2)}

Training set: 2000 samples
Test set:     2000 samples
Features:     ['age', 'income', 'clicks', 'purchase_amount']


## User Classification

Train a classifier to predict the user category (`User1`, `User2`, `User3`),
which serves as the **context** for the contextual bandit.


In [4]:
# --- User Classification ---
# Train a Random Forest classifier to predict user category (context)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train_scaled, y_train_enc)

# Predict on training and test sets
y_train_pred = clf.predict(X_train_scaled)
y_test_pred = clf.predict(X_test_scaled)

# Evaluate
train_acc = accuracy_score(y_train_enc, y_train_pred)
test_acc = accuracy_score(y_test_enc, y_test_pred)

print(f"Training Accuracy: {train_acc:.4f}")
print(f"Test Accuracy:     {test_acc:.4f}")

print("\n--- Classification Report (Test Set) ---")
print(classification_report(y_test_enc, y_test_pred, target_names=le.classes_))

# Feature importance
importances = clf.feature_importances_
for feat, imp in sorted(zip(feature_cols, importances), key=lambda x: -x[1]):
    print(f"  {feat:20s}: {imp:.4f}")

Training Accuracy: 1.0000
Test Accuracy:     0.3115

--- Classification Report (Test Set) ---
              precision    recall  f1-score   support

       user1       0.30      0.32      0.31       672
       user2       0.31      0.31      0.31       679
       user3       0.33      0.31      0.32       649

    accuracy                           0.31      2000
   macro avg       0.31      0.31      0.31      2000
weighted avg       0.31      0.31      0.31      2000

  purchase_amount     : 0.2885
  income              : 0.2838
  clicks              : 0.2238
  age                 : 0.2038


# `Contextual Bandit`

## Reward Sampler Initialization

The sampler is initialized using the student's roll number `i`.
Rewards are obtained using `sampler.sample(j)`.


## Arm Mapping

| Arm Index (j) | News Category | User Context |
|--------------|---------------|--------------|
| 0–3          | Entertainment, Education, Tech, Crime | User1 |
| 4–7          | Entertainment, Education, Tech, Crime | User2 |
| 8–11         | Entertainment, Education, Tech, Crime | User3 |

## Epsilon-Greedy Strategy

This section implements the epsilon-greedy contextual bandit algorithm.


## Upper Confidence Bound (UCB)

This section implements the UCB strategy for contextual bandits.

## SoftMax Strategy

This section implements the SoftMax strategy with temperature $ \tau = 1$.


## Reinforcement Learning Simulation

We simulate the bandit algorithms for $T = 10,000$ steps and record rewards.

P.S.: Change $T$ value as and if required.


## Results and Analysis

This section presents:
- Average Reward vs Time
- Hyperparameter comparisons
- Observations and discussion


## Final Observations

- Comparison of Epsilon-Greedy, UCB, and SoftMax
- Effect of hyperparameters
- Strengths and limitations of each approach
